In [2]:
import pandas as pd
import numpy as np
   

In [3]:
df=pd.read_csv('cleaned_merged_seasons.csv')
df.head()

C:\Users\Seif El Din Nassar\AppData\Local\Temp\ipykernel_25420\363551071.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('cleaned_merged_seasons.csv')


,season_x,name,position,team_x,assists,bonus,bps,clean_sheets,creativity,element,...,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW
0,2016-17,Aaron Cresswell,DEF,NaN,0,0,0,0,0.0,454,...,2.0,0.0,0,0,0,0,55,False,0,1
1,2016-17,Aaron Lennon,MID,NaN,0,0,6,0,0.3,142,...,1.0,0.0,1,0,0,0,60,True,0,1
2,2016-17,Aaron Ramsey,MID,NaN,0,0,5,0,4.9,16,...,3.0,23.0,2,0,0,0,80,True,0,1
3,2016-17,Abdoulaye Doucouré,MID,NaN,0,0,0,0,0.0,482,...,1.0,0.0,0,0,0,0,50,False,0,1
4,2016-17,Adam Forshaw,MID,NaN,0,0,3,0,1.3,286,...,1.0,0.0,1,0,0,0,45,True,1,1


  season_x  fixture                opp_team_name
0  2016-17        1           [Swansea, Burnley]
1  2016-17        2  [West Brom, Crystal Palace]
2  2016-17        3             [Spurs, Everton]
3  2016-17        4            [Leicester, Hull]
4  2016-17        5       [Sunderland, Man City]


In [7]:
def fill_team_x(row):
    df_grouped = df.groupby(['season_x', 'fixture'])['opp_team_name'].unique().reset_index()
    df_grouped['opp_team_name'] = df_grouped['opp_team_name'].apply(list)
    if pd.isnull(row['team_x']):
        teams = row['opp_team_name']
        # Find all teams in the fixture
        all_teams = df_grouped.loc[
            (df_grouped['season_x'] == row['season_x']) & 
            (df_grouped['fixture'] == row['fixture']),
            'opp_team_name'
        ].values[0]
        # The other team is the one not equal to opp_team_name
        other_team = [t for t in all_teams if t != row['opp_team_name']]
        return other_team[0] if other_team else row['team_x']
    return row['team_x']

df['team_x'] = df.apply(fill_team_x, axis=1)

KeyboardInterrupt: 

In [4]:
df_copy=df.copy()

In [5]:
df_grouped = df.groupby(['season_x', 'fixture'])['opp_team_name'].unique().reset_index()
df_grouped['opp_team_name'] = df_grouped['opp_team_name'].apply(list)
def fill_team_x_vectorized(df, df_grouped):
    
    # Create a mapping from (season_x, fixture) to list of teams
    fixture_team_map = {(row['season_x'], row['fixture']): row['opp_team_name'] for _, row in df_grouped.iterrows()}
    
    # Only fill where team_x is null
    mask = df['team_x'].isnull()
    # Get all teams for each row's fixture
    all_teams = df.loc[mask].apply(lambda row: fixture_team_map.get((row['season_x'], row['fixture']), []), axis=1)
    # Find the other team
    other_team = [
        [t for t in teams if t != row['opp_team_name']][0] if len(teams) == 2 else np.nan
        for teams, (_, row) in zip(all_teams, df.loc[mask].iterrows())
    ]
    # Fill missing team_x
    df.loc[mask, 'team_x'] = other_team
    return df

df_copy = fill_team_x_vectorized(df_copy, df_grouped)

In [6]:
df_copy.head()

,season_x,name,position,team_x,assists,bonus,bps,clean_sheets,creativity,element,...,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW
0,2016-17,Aaron Cresswell,DEF,West Ham,0,0,0,0,0.0,454,...,2.0,0.0,0,0,0,0,55,False,0,1
1,2016-17,Aaron Lennon,MID,Everton,0,0,6,0,0.3,142,...,1.0,0.0,1,0,0,0,60,True,0,1
2,2016-17,Aaron Ramsey,MID,Arsenal,0,0,5,0,4.9,16,...,3.0,23.0,2,0,0,0,80,True,0,1
3,2016-17,Abdoulaye Doucouré,MID,Watford,0,0,0,0,0.0,482,...,1.0,0.0,0,0,0,0,50,False,0,1
4,2016-17,Adam Forshaw,MID,Middlesbrough,0,0,3,0,1.3,286,...,1.0,0.0,1,0,0,0,45,True,1,1


In [8]:
df_copy.isnull().sum()

season_x             0
name                 0
position             0
team_x               0
assists              0
bonus                0
bps                  0
clean_sheets         0
creativity           0
element              0
fixture              0
goals_conceded       0
goals_scored         0
ict_index            0
influence            0
kickoff_time         0
minutes              0
opponent_team        0
opp_team_name        0
own_goals            0
penalties_missed     0
penalties_saved      0
red_cards            0
round                0
saves                0
selected             0
team_a_score         0
team_h_score         0
threat               0
total_points         0
transfers_balance    0
transfers_in         0
transfers_out        0
value                0
was_home             0
yellow_cards         0
GW                   0
dtype: int64

In [9]:
df_copy.to_csv('filled_teams_seasons.csv', index=False)